# 02 (Joint) — Text cleaning + features for FineFake + FakeNewsNet
MAS Lab 2 — Fake news detection (WCO + MOA) — **joint text-only model**

Goal: **one general detector** that learned from both datasets and can classify any news text.
Only the **text** is shared by both datasets (FakeNewsNet has no images or knowledge vectors), so this model is **text-only**.

- **FineFake**: the texts already cleaned in notebook 02a, with **the same train/test split** (its test news stay unseen)
- **FakeNewsNet**: titles downloaded from the official repository, cleaned with the same rules + removal of outlet names

**Inputs:** attach the `02a_finefake_cleaning` output. **Accelerator:** GPU T4. **Internet:** ON.

**Output** (`features/`, same file names as before so notebooks 03 and 04 run unchanged): `X_text.npy`, `X_text_pca.npy`, `y.npy`, `is_train.npy`, `meta.pkl`, `pca.pkl`, `pca_info.json`.

## 0. Setup

In [1]:
!pip install -q sentence-transformers
import os, re, html, glob, json
import numpy as np
import pandas as pd
import torch

SEED = 42
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
OUT = '/kaggle/working/features'
os.makedirs(OUT, exist_ok=True)
print('Device:', DEVICE)

Device: cuda


## 1. FineFake: cleaned texts and split from notebook 02a

In [2]:
found = glob.glob('/kaggle/input/**/finefake_clean.pkl', recursive=True)
assert found, 'attach the 02a_finefake_cleaning output (Add Input)'
ff = pd.read_pickle(found[0])
ff = ff[['text_clean', 'label', 'platform', 'split']].copy()
ff['dataset'] = 'finefake'
print('FineFake:', ff.shape)
print(pd.crosstab(ff['split'], ff['label']))

FineFake: (15298, 5)
label     0     1
split            
test   1599  1461
train  6394  5844


## 2. FakeNewsNet: download

In [3]:
BASE = 'https://raw.githubusercontent.com/KaiDMML/FakeNewsNet/master/dataset'
parts = []
for source in ['politifact', 'gossipcop']:
    for label, name in [(0, 'fake'), (1, 'real')]:
        d = pd.read_csv(f'{BASE}/{source}_{name}.csv')
        d['platform'], d['label'] = source, label
        parts.append(d)
fn = pd.concat(parts, ignore_index=True)
print('FakeNewsNet raw:', fn.shape)
print(fn.groupby(['platform', 'label']).size().rename('count'))
print('\nTitle length (words):', fn['title'].astype(str).str.split().str.len().describe().round(1).to_dict())

FakeNewsNet raw: (23196, 6)
platform    label
gossipcop   0         5323
            1        16817
politifact  0          432
            1          624
Name: count, dtype: int64

Title length (words): {'count': 23196.0, 'mean': 11.2, 'std': 4.0, 'min': 1.0, '25%': 9.0, '50%': 11.0, '75%': 14.0, 'max': 53.0}


## 3. FakeNewsNet: cleaning
Same rules as FineFake (HTML, quotes, URLs, mentions, hashtags, whitespace) + removal of **outlet names** written in titles (`... | Miami Herald`, `Opinion | ...`, `... - Gurl.com`), which reveal the source.
`news_url` and `tweet_ids` are dropped (the URL domain would be a source shortcut).

In [4]:
URL_RE     = re.compile(r'https?://\S+|www\.\S+')
MENTION_RE = re.compile(r'@\w+')
HASHTAG_RE = re.compile(r'#(\w+)')
HTML_RE    = re.compile(r'<[^>]+>')
QUOTES = {'\u2019': "'", '\u2018': "'", '\u201c': '"', '\u201d': '"', '\u2013': '-', '\u2014': '-'}
OPINION_RE    = re.compile(r'^\s*opinion\s*\|\s*', re.IGNORECASE)
OUTLET_END_RE = re.compile(r'\s*\|.*$')
DOMAIN_END_RE = re.compile(r'\s+-\s+\S+\.(com|net|org|co\.uk)\s*$', re.IGNORECASE)

def clean_title(t):
    t = html.unescape(str(t))
    t = HTML_RE.sub(' ', t)
    for k, v in QUOTES.items():
        t = t.replace(k, v)
    t = OPINION_RE.sub('', t)
    t = OUTLET_END_RE.sub('', t)
    t = DOMAIN_END_RE.sub('', t)
    t = URL_RE.sub(' [URL] ', t)
    t = MENTION_RE.sub(' ', t)
    t = HASHTAG_RE.sub(r'\1', t)
    return re.sub(r'\s+', ' ', t).strip()

fn['text_clean'] = fn['title'].apply(clean_title)
for i in fn.index[fn['title'].astype(str).str.contains(r'\||#|&\w+;', regex=True)][:4]:
    print('BEFORE:', repr(fn.at[i, 'title']));  print('AFTER :', repr(fn.at[i, 'text_clean'])); print()

fn = fn[fn['text_clean'].str.split().str.len() >= 3].copy()
key = fn['text_clean'].str.lower()
conflicts = fn.groupby(key)['label'].nunique()
fn = fn[~key.isin(conflicts[conflicts > 1].index)]
fn = fn.loc[~fn['text_clean'].str.lower().duplicated(keep='first')]

in_ff = fn['text_clean'].str.lower().isin(set(ff['text_clean'].str.lower()))
print('FakeNewsNet titles also present in FineFake (removed):', int(in_ff.sum()))
fn = fn[~in_ff][['text_clean', 'label', 'platform']].reset_index(drop=True)
fn['dataset'] = 'fakenewsnet'
print('FakeNewsNet clean:', fn.shape)
print(fn.groupby(['platform', 'label']).size().rename('count'))

BEFORE: 'Putin says: ‘Pope Francis Is Not A Man Of God’ | Must-See\xa0!!'
AFTER : "Putin says: 'Pope Francis Is Not A Man Of God'"

BEFORE: 'UPDATE: Florida Governor Rick Scott Now Listed As &#8216;Critical&#8217; After Bizarre Hurricane Cleanup Accident'
AFTER : "UPDATE: Florida Governor Rick Scott Now Listed As 'Critical' After Bizarre Hurricane Cleanup Accident"

BEFORE: 'Clinton Body Count Series #5: Woman Set to Testify Against Clintons Blown Up in Home Explosion'
AFTER : 'Clinton Body Count Series 5: Woman Set to Testify Against Clintons Blown Up in Home Explosion'

BEFORE: 'NRA President David Keene Rejects White House Gun Control Approach | PBS NewsHour | Jan. 15, 2013'
AFTER : 'NRA President David Keene Rejects White House Gun Control Approach'

FakeNewsNet titles also present in FineFake (removed): 0
FakeNewsNet clean: (21218, 4)
platform    label
gossipcop   0         4700
            1        15569
politifact  0          415
            1          534
Name: count, dtype: in

## 4. FakeNewsNet train/test split (80/20, stratified on source × label)

In [5]:
from sklearn.model_selection import train_test_split
strata = fn['platform'] + '_' + fn['label'].astype(str)
idx_tr, idx_te = train_test_split(fn.index, test_size=0.2, stratify=strata, random_state=SEED)
fn['split'] = 'train'
fn.loc[idx_te, 'split'] = 'test'
print(pd.crosstab([fn['platform'], fn['label']], fn['split']))

split             test  train
platform   label             
gossipcop  0       940   3760
           1      3114  12455
politifact 0        83    332
           1       107    427


## 5. Merge the two datasets

In [6]:
df = pd.concat([ff, fn], ignore_index=True)
print('Joint dataset:', df.shape)
print(pd.crosstab([df['dataset'], df['split']], df['label'], margins=True))
print(f"\nShare of fake news: {1 - df['label'].mean():.1%}  (class imbalance handled by macro-F1 + stratified folds)")

Joint dataset: (36516, 5)
label                  0      1    All
dataset     split                     
fakenewsnet test    1023   3221   4244
            train   4092  12882  16974
finefake    test    1599   1461   3060
            train   6394   5844  12238
All                13108  23408  36516

Share of fake news: 35.9%  (class imbalance handled by macro-F1 + stratified folds)


## 6. Text embeddings (`all-mpnet-base-v2`, same model as before)

In [7]:
from sentence_transformers import SentenceTransformer
text_model = SentenceTransformer('all-mpnet-base-v2', device=DEVICE)
text_model.max_seq_length = 384
X_text = text_model.encode(df['text_clean'].tolist(), batch_size=64, show_progress_bar=True,
                           convert_to_numpy=True, normalize_embeddings=True).astype(np.float32)
print('X_text:', X_text.shape, '| NaN:', np.isnan(X_text).sum())

q = int(df.index[df['dataset'] == 'fakenewsnet'][0])
sims = X_text @ X_text[q]
print('\nQUERY:', df.at[q, 'text_clean'], '|', df.at[q, 'dataset'], '| label', df.at[q, 'label'])
for i in np.argsort(-sims)[1:4]:
    print(f'{sims[i]:.3f}:', df.at[i, 'text_clean'][:110], '|', df.at[i, 'dataset'], '| label', df.at[i, 'label'])

X_text: (36516, 768) | NaN: 0

QUERY: BREAKING: First NFL Team Declares Bankruptcy Over Kneeling Thugs | fakenewsnet | label 0
0.565: During the 2023 NFL season, referees ejected four NFL players from a game for kneeling during the national ant | finefake | label 0
0.554: BREAKING: President Trump Signs Executive Order Stripping NFL Of 'Non-Profit' Status | fakenewsnet | label 0
0.534: In October 2023, National Football League coaches unanimously voted to ban players from kneeling during the U. | finefake | label 0


## 7. PCA fitted on the joint train rows only (768 → 128)

In [8]:
from sklearn.decomposition import PCA
import joblib
is_train = (df['split'] == 'train').to_numpy()
pca = PCA(n_components=128, random_state=SEED).fit(X_text[is_train])
X_text_pca = pca.transform(X_text).astype(np.float32)
var = float(np.cumsum(pca.explained_variance_ratio_)[-1])
print(f'768 -> 128 dims | variance kept: {var:.1%}')
joblib.dump(pca, f'{OUT}/pca.pkl')

768 -> 128 dims | variance kept: 74.0%


['/kaggle/working/features/pca.pkl']

## 8. Save (same file names as the other feature folders)

In [9]:
np.save(f'{OUT}/X_text.npy', X_text)
np.save(f'{OUT}/X_text_pca.npy', X_text_pca)
np.save(f'{OUT}/y.npy', df['label'].to_numpy().astype(np.int8))
np.save(f'{OUT}/is_train.npy', is_train)
df[['text_clean', 'label', 'platform', 'dataset', 'split']].to_pickle(f'{OUT}/meta.pkl')
json.dump({'dataset': 'FineFake + FakeNewsNet (text only)', 'text_model': 'all-mpnet-base-v2',
           'rows': int(len(df)), 'pca': {'text': {'n_components': 128, 'explained_variance': round(var, 4)}}},
          open(f'{OUT}/pca_info.json', 'w'), indent=2)
for f in sorted(os.listdir(OUT)):
    print(f'{f:20s} {os.path.getsize(f"{OUT}/{f}") / 1e6:6.1f} MB')

X_text.npy            112.2 MB
X_text_pca.npy         18.7 MB
is_train.npy            0.0 MB
meta.pkl               23.8 MB
pca.pkl                 0.4 MB
pca_info.json           0.0 MB
y.npy                   0.0 MB
